In [15]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [16]:
# Load Data
df = pd.read_csv("../data/full/data_full.csv")

# --- Preprocessing ---

# 1. SES: Low vs High
# economically_disadvantaged_1 = Low SES (Free/Reduced Lunch)
# economically_disadvantaged_0 = High SES (Likely Paid Lunch/Not Disadvantaged)
df['ses_group'] = df['economically_disadvantaged_1'].apply(lambda x: 'Low SES' if x == 1 else 'High SES')

# 2. Race: White vs Non-White
df['race_group'] = df['race_ethnicity_White'].apply(lambda x: 'White' if x == 1 else 'Non-White')

# 3. Gender
# Assuming gender_F is reliable for Female.
def get_gender(row):
    if row['gender_F'] == 1:
        return 'Female'
    else:
        return 'Male'
        
df['gender_group'] = df.apply(get_gender, axis=1)

# --- Selection of Metrics ---
# We use existing columns from the dataset instead of calculating new ones.

# Dictionary mapping column names to readable labels/descriptions
metric_descriptions = {
    'holistic_essay_score': 'Holistic Essay Score (Outcome)',
    
    # Length / Volume Metrics
    'taassc_nwords': 'Word Count (Volume)',
    
    # Lexical Complexity (Reading Level Proxies)
    'taassc_wrd_length': 'Avg Word Length (Lexical Sophistication)',
    
    # Syntactic Complexity (Style)
    'taassc_mlc': 'Mean Length of Clause (Syntactic Complexity)',
    'taassc_mltu': 'Mean Length of T-Unit (Sentence Complexity)',
    'taassc_mean_verbal_deps': 'Mean Verbal Dependencies',
    'taassc_infinitive_prop': 'Proportion of Infinitives',
    'taassc_nonfinite_prop': 'Proportion of Non-Finite Clauses'
}

target_metrics = list(metric_descriptions.keys())

print("Data loaded and groups defined. Using metrics:")
for m, desc in metric_descriptions.items():
    print(f" - {m}: {desc}")

Data loaded and groups defined. Using metrics:
 - holistic_essay_score: Holistic Essay Score (Outcome)
 - taassc_nwords: Word Count (Volume)
 - taassc_wrd_length: Avg Word Length (Lexical Sophistication)
 - taassc_mlc: Mean Length of Clause (Syntactic Complexity)
 - taassc_mltu: Mean Length of T-Unit (Sentence Complexity)
 - taassc_mean_verbal_deps: Mean Verbal Dependencies
 - taassc_infinitive_prop: Proportion of Infinitives
 - taassc_nonfinite_prop: Proportion of Non-Finite Clauses


In [17]:
def get_group_stats(df, group_col, group_label_map, metrics):
    results = {}
    
    # group_label_map: {'GroupValue': 'DisplayName'}
    # e.g., {'Low SES': 'Low SES', 'High SES': 'High SES'}
    
    for group_val, display_name in group_label_map.items():
        sub_df = df[df[group_col] == group_val]
        n = len(sub_df)
        stats = sub_df[metrics].mean().to_dict()
        
        # Add relevant metadata
        col_data = {'N': n}
        for m in metrics:
            col_data[f"{m}_mean"] = stats[m]
            col_data[f"{m}_sd"] = sub_df[metrics].std()[m]
            
        results[display_name] = col_data
        
    return pd.DataFrame(results)

# 1. Full Sample
full_stats = pd.DataFrame(index=['N'] + [f"{m}_{s}" for m in target_metrics for s in ['mean', 'sd']])
full_stats['Full Sample'] = pd.Series({'N': len(df)})
for m in target_metrics:
    full_stats.loc[f"{m}_mean", 'Full Sample'] = df[m].mean()
    full_stats.loc[f"{m}_sd", 'Full Sample'] = df[m].std()

# 2. SES Breakdown
ses_stats = get_group_stats(df, 'ses_group', {'Low SES': 'Low SES', 'High SES': 'High SES'}, target_metrics)

# 3. Race Breakdown
race_stats = get_group_stats(df, 'race_group', {'White': 'White', 'Non-White': 'Non-White'}, target_metrics)

# 4. Gender Breakdown
gender_stats = get_group_stats(df, 'gender_group', {'Female': 'Female', 'Male': 'Male'}, target_metrics)

# Combine all
final_table = pd.concat([full_stats, ses_stats, race_stats, gender_stats], axis=1)

# Calculate Gaps
# SES Gap (High - Low)
final_table['Gap SES (High-Low)'] = final_table['High SES'] - final_table['Low SES']

# Race Gap (White - Non-White)
final_table['Gap Race (White-NonWhite)'] = final_table['White'] - final_table['Non-White']

# Gender Gap (F - M)
final_table['Gap Gender (F-M)'] = final_table['Female'] - final_table['Male']

# Reorder for display
cols_order = [
    'Full Sample', 
    'Low SES', 'High SES', 'Gap SES (High-Low)',
    'Non-White', 'White', 'Gap Race (White-NonWhite)',
    'Male', 'Female', 'Gap Gender (F-M)'
]
final_table = final_table[cols_order]

# Display
final_table.round(3)

,Full Sample,Low SES,High SES,Gap SES (High-Low),Non-White,White,Gap Race (White-NonWhite),Male,Female,Gap Gender (F-M)
N,20759.000,9643.000,11116.000,1473.000,11371.000,9388.000,-1983.000,10133.000,10626.000,493.000
holistic_essay_score_mean,3.337,2.979,3.648,0.669,3.224,3.473,0.249,3.219,3.449,0.230
holistic_essay_score_sd,1.172,1.063,1.173,0.111,1.179,1.148,-0.031,1.162,1.171,0.009
taassc_nwords_mean,412.621,368.372,451.006,82.635,405.240,421.561,16.321,396.152,428.325,32.173
taassc_nwords_sd,194.214,172.395,203.689,31.294,197.084,190.308,-6.777,190.907,196.041,5.134
taassc_wrd_length_mean,4.346,4.295,4.390,0.095,4.353,4.337,-0.016,4.321,4.370,0.048
taassc_wrd_length_sd,0.326,0.312,0.331,0.020,0.324,0.328,0.004,0.328,0.322,-0.005
taassc_mlc_mean,12.128,11.911,12.316,0.405,12.067,12.202,0.134,12.165,12.093,-0.073
taassc_mlc_sd,2.996,3.102,2.888,-0.214,3.056,2.921,-0.135,3.090,2.905,-0.185
taassc_mltu_mean,28.954,28.721,29.157,0.436,29.400,28.414,-0.986,29.654,28.287,-1.366


In [ ]:
# Export to CSV just in case
final_table.to_csv("descriptive_statistics_summary.csv")

# Print descriptions again for context in logical place
print("Column Descriptions included in analysis:")
for m, desc in metric_descriptions.items():
    print(f" - {m}: {desc}")
    
print("\nTable saved to descriptive_statistics_summary.csv")